# Verificación de la puerta P2 sobre la imputación — modelo perinatal NCHS (versión 2b)

**Por qué.** En la versión 2b el AUROC es 0,7455 con todos los registros, pero 0,689 cuando las mismas predicciones se restringen a los registros completos. Parte de la brecha viene de quiénes quedan (los registros completos conservan el 76–81% de las muertes fetales), pero hay una segunda razón posible: la imputación por la mediana convierte cada desconocido en un mismo valor repetido, y XGBoost puede aislar ese valor, que funcionaría como un indicador oculto de "desconocido". Los desconocidos son mucho más frecuentes en el reporte de muerte fetal, así que sería la señal del tipo de formulario que identificaron la puerta P2 y el modelo M6. Este notebook lo verifica **antes** de construir la versión 3.

**Diseño (misma muestra, semilla, particiones, hiperparámetros y recalibración del notebook 09).** Cuatro estrategias de imputación para XGBoost, comparadas en las mismas particiones:
- **mediana** (como las versiones 1 a 2b);
- **sorteo aleatorio**: cada desconocido se reemplaza por un valor sorteado entre los valores observados de la parte de entrenamiento, ponderado a la población; el valor imputado no deja rastro de haber sido desconocido;
- **mediana + indicadores**: la señal del formulario hecha explícita (cota superior);
- **nativa**: el manejo propio de faltantes de XGBoost (diagnóstico del notebook 05).

Además mide si los datos imputados revelan qué registros tenían un desconocido (AUROC para "el registro tenía un desconocido").

**Regla de decisión, fijada antes de correr.** Si el AUROC con sorteo aleatorio queda más de 0,005 por debajo del de la mediana, con el IC 95% de la diferencia por debajo de 0, la imputación por la mediana lleva una señal del formulario importante y **la versión 3 usa imputación por sorteo aleatorio**. Si no, la versión 3 conserva la mediana.

Salida: `data/outputs_10_p2_imputation_check.txt`. Requiere el notebook 09 (tabla contrato `perinatal_nchs_v2b` y su salida). Unos 30 a 60 minutos. Versión en inglés: `10_p2_imputation_check.ipynb` (mismo código).

In [1]:
import os, re, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONTRACT_DIR = Path(os.environ.get("NCHS_CONTRACT4_DIR", ROOT / "data" / "contract" / "perinatal_nchs_v2b"))
OUT_FILE = ROOT / "data" / "outputs_10_p2_imputation_check.txt"
SEED = 20261121                       # same seed, sample and folds as notebook 09
YEARS, DEV_YEARS = list(range(2016, 2024)), list(range(2016, 2022))
FRACTION = float(os.environ.get("NCHS_FRACTION", 0.02))
QUICK = os.environ.get("NCHS_QUICK") == "1"
N_JOBS = int(os.environ.get("NCHS_JOBS", -1))
FOLDS, B = (3, 30) if QUICK else (5, 200)
pd.set_option("display.max_colwidth", None)
lines = []
def show(title, obj):
    txt = obj.to_string() if hasattr(obj, "to_string") else str(obj)
    block = f"\n== {title}\n{txt}"
    print(block); lines.append(block)

# Predictors of version 2b (notebook 09) and the hyperparameters its random search chose for XGBoost
DROP = ["unit_id", "group_id", "period", "component", "ga_weeks", "label_main", "label_fetal",
        "label_early_neonatal", "estofd", "label_ge28", "n_unknown", "father_age"]
PREDICTORS = [c for c in pd.read_parquet(CONTRACT_DIR / "year_2023.parquet").columns if c not in DROP]
assert len(PREDICTORS) == 35, len(PREDICTORS)
out09 = (ROOT / "data" / "outputs_09_model_v2b.txt").read_text(encoding="utf-8")
XGB_PARAMS = json.loads(re.search(r"^XGB\s+(\{.*?\})\s+[\d.]+\s+[\d.]+\s*$", out09, re.M).group(1))
if QUICK:
    XGB_PARAMS["n_estimators"] = 50
show("Version 2b: predictors and XGBoost hyperparameters (from data/outputs_09_model_v2b.txt)",
     pd.Series({"parameters": len(PREDICTORS), "hyperparameters": json.dumps(XGB_PARAMS)}))


== Version 2b: predictors and XGBoost hyperparameters (from data/outputs_09_model_v2b.txt)
parameters                                                                                                                                                                                           35
hyperparameters    {"max_depth": 6, "learning_rate": 0.027, "min_child_weight": 1.3367, "subsample": 0.697, "colsample_bytree": 0.8543, "reg_lambda": 8.8833, "reg_alpha": 0.8853, "n_estimators": 337}


## 1. Muestra de desarrollo

In [2]:
# Development sample exactly as in notebook 09: every event and FRACTION of non-events, weight 1/FRACTION.
rng = np.random.default_rng(SEED); parts = []
for y in DEV_YEARS:
    c = pd.read_parquet(CONTRACT_DIR / f"year_{y}.parquet",
                        columns=PREDICTORS + ["label_main", "label_fetal", "label_early_neonatal", "component"])
    parts.append(c[c["label_main"].to_numpy() | (rng.random(len(c)) < FRACTION)])
dev = pd.concat(parts, ignore_index=True); del parts
if QUICK:
    dev = dev.sample(n=min(len(dev), 40000), random_state=SEED).reset_index(drop=True)
X = dev[PREDICTORS].astype("float32").to_numpy()
y = dev["label_main"].to_numpy(bool)
w = np.where(y, 1.0, 1.0 / FRACTION)
fetal = dev["label_fetal"].to_numpy(bool); early = dev["label_early_neonatal"].to_numpy(bool)
is_fd_record = dev["component"].astype("string").eq("fetal_death").to_numpy()
incomplete = np.isnan(X).any(axis=1)
show("Development sample (2016-2021, all records)", pd.Series({
    "rows": len(dev), "events": int(y.sum()), "fetal deaths": int(fetal.sum()), "early neonatal deaths": int(early.sum()),
    "% records with an unknown, non-events (weighted)": round(100 * np.average(incomplete[~y], weights=w[~y]), 1),
    "% records with an unknown, fetal deaths": round(100 * incomplete[fetal].mean(), 1),
    "% records with an unknown, early neonatal deaths": round(100 * incomplete[early].mean(), 1)}))


== Development sample (2016-2021, all records)
rows                                                600442.0
events                                              149057.0
fetal deaths                                        102529.0
early neonatal deaths                                46528.0
% records with an unknown, non-events (weighted)        13.8
% records with an unknown, fetal deaths                 46.9
% records with an unknown, early neonatal deaths        31.8


## 2. Estrategias de imputación y validación cruzada

In [3]:
# Imputation strategies compared (fixed before running; see the decision rule in the introduction):
#   median        median of the training part (as notebooks 05 to 09): every unknown becomes one repeated value
#   random draw   each unknown replaced by a value drawn at random from the observed values of the training part,
#                 weighted by the sample weights (so the draws follow the population, not the case-control sample).
#                 The imputed value carries no trace of having been unknown.
#   median + ind  median plus one missingness indicator per variable: the form signal made explicit (upper bound)
#   native        XGBoost's own handling of missing values (diagnostic of notebook 05)
KINDS = ["median", "random draw", "median + ind", "native"]

def fit_imputer(kind, Xf, wf, seed):
    has_na = np.isnan(Xf).any(axis=0)
    if kind == "native":
        return lambda Xn: Xn
    med = np.nanmedian(Xf, axis=0)
    if kind in ("median", "median + ind"):
        def f(Xn):
            Z = np.where(np.isnan(Xn), med, Xn)
            return np.hstack([Z, np.isnan(Xn[:, has_na]).astype("float32")]) if kind == "median + ind" else Z
        return f
    pools = []
    for j in range(Xf.shape[1]):
        ok = ~np.isnan(Xf[:, j]); pools.append((Xf[ok, j], wf[ok] / wf[ok].sum()))
    def f(Xn):
        Z = Xn.copy(); r = np.random.default_rng(seed)
        for j, (vals, pw) in enumerate(pools):
            m = np.isnan(Z[:, j])
            if m.any():
                Z[m, j] = r.choice(vals, size=int(m.sum()), p=pw)
        return Z
    return f

logit = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))

def fit_calibrated(kind, Xtr, ytr, wtr, seed):
    """As in notebooks 05 to 09: fit on 80% of the training part, Platt recalibration on the other 20%."""
    Xf, Xc, yf, yc, wf, wc = train_test_split(Xtr, ytr, wtr, test_size=0.2, stratify=ytr, random_state=seed)
    imp = fit_imputer(kind, Xf, wf, seed)
    m = XGBClassifier(tree_method="hist", n_jobs=N_JOBS, random_state=SEED, **XGB_PARAMS)
    m.fit(imp(Xf), yf, sample_weight=wf)
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(m.predict_proba(imp(Xc))[:, 1]).reshape(-1, 1), yc, sample_weight=wc)
    return lambda Xn: platt.predict_proba(logit(m.predict_proba(imp(Xn))[:, 1]).reshape(-1, 1))[:, 1]

# Out-of-fold predictions with the same folds for every strategy (paired comparison)
folds = list(StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(X, y))
oof = {k: np.zeros(len(y)) for k in KINDS}
for k in KINDS:
    t = time.time()
    for i, (tr, te) in enumerate(folds):
        oof[k][te] = fit_calibrated(k, X[tr], y[tr], w[tr], SEED + i)(X[te])
    print(f"{k}: {(time.time() - t) / 60:.1f} min")

median: 1.0 min
random draw: 1.1 min
median + ind: 1.2 min
native: 2.5 min


## 3. Resultados y decisión

In [4]:
# Results. AUROC in the whole sample and in subsets; paired stratified bootstrap of the differences with the median.
non_event = ~y
subsets = {"all records": (np.ones(len(y), bool), y),
           "fetal deaths vs non-events": (fetal | non_event, fetal),
           "early neonatal deaths vs non-events": (early | non_event, early),
           "complete records": (~incomplete, y),
           "records with an unknown": (incomplete, y)}
tab = pd.DataFrame({k: {s: round(roc_auc_score(yy[m], oof[k][m]), 4) for s, (m, yy) in subsets.items()} for k in KINDS})
show(f"AUROC by imputation strategy ({FOLDS}-fold cross-validation, recalibrated, same folds)", tab)

rb = np.random.default_rng(SEED + 9); pos, neg = np.where(y)[0], np.where(~y)[0]
diffs = {k: [] for k in KINDS if k != "median"}
for _ in range(B):
    ix = np.concatenate([rb.choice(pos, len(pos)), rb.choice(neg, len(neg))])
    a0 = roc_auc_score(y[ix], oof["median"][ix])
    for k in diffs:
        diffs[k].append(roc_auc_score(y[ix], oof[k][ix]) - a0)
dtab = pd.DataFrame({k: {"difference vs median": round(float(np.mean(v)), 4), "2.5%": round(float(np.percentile(v, 2.5)), 4),
                         "97.5%": round(float(np.percentile(v, 97.5)), 4)} for k, v in diffs.items()}).T
show("AUROC difference with median imputation, all records (paired bootstrap, 95% CI)", dtab)

# Can the imputed data reveal which records had an unknown? AUROC of XGBoost predicting 'any unknown'
# from the imputed predictors (3 folds, unweighted). Near 0.5: the imputation hides the missingness.
rec = {}
for k in ["median", "random draw"]:
    p = np.zeros(len(y))
    for i, (tr, te) in enumerate(StratifiedKFold(3, shuffle=True, random_state=SEED).split(X, incomplete)):
        imp = fit_imputer(k, X[tr], w[tr], SEED + i)
        m = XGBClassifier(tree_method="hist", n_jobs=N_JOBS, random_state=SEED, n_estimators=100, max_depth=6, learning_rate=0.1)
        m.fit(imp(X[tr]), incomplete[tr]); p[te] = m.predict_proba(imp(X[te]))[:, 1]
    rec[k] = round(roc_auc_score(incomplete, p), 4)
show("Recoverability of missingness: AUROC for 'record had an unknown' from the imputed predictors", pd.Series(rec))

# Decision rule fixed before running (introduction of this notebook)
d, lo = dtab.loc["random draw", "difference vs median"], dtab.loc["random draw", "2.5%"]
material = (-d > 0.005) and (dtab.loc["random draw", "97.5%"] < 0)
show("Decision (gate G2, imputation)", pd.Series({
    "AUROC median": tab.loc["all records", "median"], "AUROC random draw": tab.loc["all records", "random draw"],
    "median imputation carries a material form signal (drop > 0.005 with CI below 0)": bool(material),
    "imputation for version 3": "random draw" if material else "median (as versions 1 to 2b)"}))
# Original objective of the thesis (sensitivity and specificity both above 0.75): the best pair any threshold can
# reach. Weighted to the full population; computed on the predictions of this notebook. Descriptive only.
def best_pairs(y, p, w):
    o = np.argsort(-p); yo, wo, po = y[o], w[o], p[o]
    tp = np.cumsum(wo * yo); fp = np.cumsum(wo * ~yo)
    sens = tp / tp[-1]; spec = 1 - fp / fp[-1]
    last = np.r_[po[1:] != po[:-1], True]          # evaluate only at the end of tied scores
    sens, spec, po = sens[last], spec[last], po[last]
    i = np.argmax(np.minimum(sens, spec))
    j = np.where(sens >= 0.75)[0][0]; k = np.where(spec >= 0.75)[0][-1]
    return {"balanced point: sensitivity": round(float(sens[i]), 3), "balanced point: specificity": round(float(spec[i]), 3),
            "balanced point: risk cut": round(float(po[i]), 4),
            "specificity when sensitivity >= 0.75": round(float(spec[j]), 3),
            "sensitivity when specificity >= 0.75": round(float(sens[k]), 3),
            "some threshold gives both >= 0.75": bool(((sens >= 0.75) & (spec >= 0.75)).any())}
show("Original objective: best sensitivity-specificity pairs (all records, 5-fold out-of-fold predictions)",
     pd.DataFrame({k: best_pairs(y, oof[k], w) for k in ["median", "random draw"]}))
OUT_FILE.write_text("Gate G2 check of the imputation (perinatal model, version 2b, development 2016-2021)\n"
                    + "\n".join(lines) + "\n", encoding="utf-8")
print("written:", OUT_FILE)


== AUROC by imputation strategy (5-fold cross-validation, recalibrated, same folds)
                                     median  random draw  median + ind  native
all records                          0.7455       0.7234        0.7699  0.7704
fetal deaths vs non-events           0.7602       0.7318        0.7925  0.7930
early neonatal deaths vs non-events  0.7131       0.7049        0.7200  0.7205
complete records                     0.6889       0.6901        0.6903  0.6909
records with an unknown              0.7386       0.7025        0.8260  0.8267

== AUROC difference with median imputation, all records (paired bootstrap, 95% CI)
              difference vs median    2.5%   97.5%
random draw                -0.0221 -0.0227 -0.0215
median + ind                0.0244  0.0238  0.0251
native                      0.0249  0.0243  0.0257

== Recoverability of missingness: AUROC for 'record had an unknown' from the imputed predictors
median         0.9130
random draw    0.7695

== Decision